In [17]:
from dotenv import load_dotenv
load_dotenv()

True

---

#### 문서 로드

In [18]:
from langchain_community.document_loaders import PDFPlumberLoader

loader = PDFPlumberLoader("../data/KCI_FI003153549_p5.pdf")
documents = loader.load()

#### 문서 분할

In [19]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
splitted_documents = text_splitter.split_documents(documents)

#### 임베딩 모델(캐싱)

In [20]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_classic.embeddings import CacheBackedEmbeddings
from langchain_classic.storage import LocalFileStore

underlying_embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

store = LocalFileStore("./cache/")

cached_embedder = CacheBackedEmbeddings.from_bytes_store(
    underlying_embeddings,
    store,
    namespace = underlying_embeddings.model
)

#### 임베딩 & FAISS(Facebook AI Similarity Search) 벡터스토어 생성 및 저장

##### Case1. In-memory

In [21]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(splitted_documents, cached_embedder)

##### Case2. 로컬 디스크 저장(기존 파일 삭제 후 저장)

In [22]:
vectorstore = FAISS.from_documents(splitted_documents, cached_embedder)

# 영구적인 파일(persistent file)**로 디스크에 저장
# 기존 폴더에 새로운 인덱스 파일을 덮어쓰기 때문에 중복된 파일이 생성되지 않음(항상 가장 마지막에 저장된 벡터스토어의 파일만 존재)
vectorstore.save_local("./faiss_index")

In [23]:
vectorstore

In [24]:
vectorstore = None

In [25]:
vectorstore

In [26]:
from langchain_community.vectorstores import FAISS

# 벡터스토어 재로딩
vectorstore = FAISS.load_local(
    "./faiss_index", # 저장된 FAISS 인덱스 폴더의 경로
    cached_embedder,
    allow_dangerous_deserialization=True, #  FAISS 인덱스 내 데이터 역직렬화(deserialization) 허용(신뢰할 수 있는 파일 일 경우)
)

In [27]:
vectorstore

##### Case3. 로컬 디스크 저장(기존 파일이 있을 경우 로드)

In [28]:
vectorstore = None

In [29]:
vectorstore

In [30]:
import os

FAISS_INDEX_PATH = "./faiss_index"

if os.path.exists(FAISS_INDEX_PATH):
    vectorstore = FAISS.load_local(
        FAISS_INDEX_PATH,
        underlying_embeddings,
        allow_dangerous_deserialization=True,
    )
else:
    # FAISS 벡터스토어 생성 및 저장
    vectorstore = FAISS.from_documents(splitted_documents, embedding_model)
    vectorstore.save_local(FAISS_INDEX_PATH)

In [31]:
vectorstore

---

### Agentic RAG

> https://docs.langchain.com/oss/python/langchain/retrieval#agentic-rag

In [32]:
# 예시 질의
query = "본 연구에서 Private LLM 구축을 위해 수집한 문서의 총 페이지 수와 문서 유형별 비율은 어떻게 되나요?"
# query = "Advance RAG 기법이 임상시험 데이터 분석에서 수행하는 주요 역할은 무엇인가요?"
# query = "본 연구에서 Private LLM 성능을 평가하기 위해 사용한 지표 3가지는 무엇인가요?"
# query = "국내에서 LLM을 임상시험에 적용한 대표적인 기관과 그 적용 사례를 2가지 이상 말해보세요."
# query = "ROUGE 평가에서 Private LLM과 ChatGPT의 Recall 값은 각각 얼마였나요?"

In [44]:
retriever = vectorstore.as_retriever()
retriever.invoke(query)

[Document(id='5dc33127-1714-41dc-b44a-b7543667100b', metadata={'source': '../data/KCI_FI003153549_p5.pdf', 'file_path': '../data/KCI_FI003153549_p5.pdf', 'page': 0, 'total_pages': 1, 'CreationDate': 'D:20250909104709', 'Creator': 'PDFium', 'Producer': 'PDFium'}, page_content='1.2 Validity of Collected Data\n본 연구에서는 의료기기 임상시험에 특화된 Private\n수집된 데이터셋은 의료기기 임상시험에 특화된\nLLM 접근 방법을 제안한다. 이 접근 방법은 도메인 특화\nPrivate LLM 구축을 위해 도메인 적합성과 다양성, 그리\n데이터셋 구축, LLM 모델 튜닝, 도메인 특화 프롬프트 적\n고 응용 가능성 측면에서 높은 타당성을 갖추고 있다. 총\n용, 그리고 도메인 특화 기능 구현의 네 가지 핵심 단계로\n111,954페이지로 구성된 데이터는 의료기기 임상시험의\n구성된다. 각 단계는 의료기기 임상시험 분야의 특수성을\n규제, 프로토콜 설계, 데이터 관리 등 전반적인 지식을 포\n반영하여 상호 유기적으로 작동하며, Figure 1와 같이 이\n괄하며, 국제 표준과 실제 임상시험 환경에서 발생할 수\n를 통해 해당 분야에서 최적의 성능을 달성하도록 설계되\n있는 다양한 시나리오를 반영하도록 설계되었다.\n었다.'),
 Document(id='7c243ea2-5fb4-46e5-aa3c-df98b30940a9', metadata={'source': '../data/KCI_FI003153549_p5.pdf', 'file_path': '../data/KCI_FI003153549_p5.pdf', 'page': 0, 'total_pages': 1, 'CreationDate': 'D:20250909104709', 'Creator

In [47]:
from langchain.tools import tool

retriever = vectorstore.as_retriever()

@tool
def search_documents(query: str) -> str:
    """의료기기 임상시험 문서에서 정보를 검색합니다. 
    전문 지식이나 연구 통계(페이지 수 등)가 필요한 경우에 사용하세요.
    """
    # 리트리버를 사용하여 문서 검색 (가장 간단한 형태)
    docs = retriever.invoke(query)
    return "\n\n".join([doc.page_content for doc in docs])

In [48]:
system_prompt = """당신은 의료기기 임상시험 전문가입니다. 

1. 정보가 필요할 경우 반드시 검색 도구(retriever_tool)를 사용하여 확인하세요.
2. 답변은 반드시 검색된 문서의 내용에만 기반하여 작성하세요.
3. 문서에 관련 내용이 없다면 억지로 꾸며내지 말고 모른다고 답변하세요.
"""

In [49]:
from langchain.agents import create_agent

agent = create_agent(
    model="google_genai:gemini-2.5-flash", 
    tools=[search_documents],
    system_prompt=system_prompt,
)

In [50]:
# 도구 호출 하지 않을 수도 있음
result = agent.invoke(
    {"messages": [{"role": "user", "content": query}]}
)

In [51]:
result

{'messages': [HumanMessage(content='본 연구에서 Private LLM 구축을 위해 수집한 문서의 총 페이지 수와 문서 유형별 비율은 어떻게 되나요?', additional_kwargs={}, response_metadata={}, id='44cb888d-289c-4327-8b06-564edd793ae8'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'search_documents', 'arguments': '{"query": "Private LLM \\uad6c\\ucd95\\uc744 \\uc704\\ud574 \\uc218\\uc9d1\\ud55c \\ubb38\\uc11c\\uc758 \\ucd1d \\ud398\\uc774\\uc9c0 \\uc218\\uc640 \\ubb38\\uc11c \\uc720\\ud615\\ubcc4 \\ube44\\uc728"}'}, '__gemini_function_call_thought_signatures__': {'8f02e1b3-e131-4727-831c-3b3b8d743ed8': 'Cu4CAXLI2nzTHUbXzLALGQPe3p3AuQMbnO666WHEwUP1Yg2O8X4BTLavHzAWMJ3DHB+84fUBhxt86n2pfEuqMQxFh5uSuirT2uwLz0s/tErVHPTJeeDMx5Txl6JceuqTbsYA7qpdjvJeEvj+T7Hlqg8u5bxif/OjUgt7w064UDaIbPqbaam9fPESEN/uoNUxvTnotiFnpIQngW0sMKJMgGThoHXFZc2qFeg3T5yYptkhXiwPH9+hanfPtNwDo5O6DiT8MBGhI1xpm53Zog1a1HcdwSMfFTRB3uw+gKzS5Pbk8DG+NpxF3O1HqJX63gZaNvBK5CDfp+QhB+h6H8QtaVa5X47wymI1A/aHjm8MLB1tb22vzLUDqnTIf5u7ypVDHBlmTMjIgU932LrA4o24ixug927Pb

In [58]:
print(result['messages'][-1].content[0]['text'])

Private LLM 구축을 위해 수집된 문서는 총 11,954페이지이며, 문서 유형별 비율은 다음과 같습니다.

*   규제 문서: 30% (FDA, EMA, PMDA 가이드라인, GCP 문서 등)
*   교육 자료: 20% (임상시험 수행자 교육 매뉴얼, 온라인 강의 자료 등)
*   프로토콜 및 보고서: 25% (임상시험 프로토콜, CSR (Clinical Study Report) 템플릿 등)
*   의료기기 특화 문서: 15% (의료기기 임상시험 계획서, 기술문서 등)
*   기타: 10% (윤리위원회 관련 문서, 환자 동의서 템플릿 등)
